Q1: Ridge Regression with Gradient Descent

Build a dataset with 7+ correlated features. Implement ridge regression from scratch using gradient descent. Test multiple learning rates (0.0001, 0.001, 0.01, 0.1, 1, 10) and regularization values (10^-15, 10^-10, 10^-5, 10^-3, 0, 1, 10, 20). Find the best combo that minimizes cost and maximizes R2.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

np.random.seed(42)
samples = 600

base = np.random.randn(samples, 1)
noise = 0.25

f1 = base + noise * np.random.randn(samples, 1)
f2 = base + noise * np.random.randn(samples, 1)
f3 = base + noise * np.random.randn(samples, 1)
f4 = 0.9 * base + 0.1 * f1 + noise * np.random.randn(samples, 1)
f5 = 0.8 * base + 0.2 * f2 + noise * np.random.randn(samples, 1)
f6 = 0.7 * base + 0.3 * f3 + noise * np.random.randn(samples, 1)
f7 = 0.6 * base + 0.4 * f1 + noise * np.random.randn(samples, 1)

X_data = np.hstack([f1, f2, f3, f4, f5, f6, f7])
weights = np.array([3.0, -2.5, 2.8, -1.9, 1.7, -1.2, 2.3])
y_data = X_data @ weights + 0.6 * np.random.randn(samples)

X_train, X_test, y_train, y_test = train_test_split(X_data, y_data, test_size=0.25, random_state=7)

sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

class MyRidge:
    def __init__(self, lr=0.01, reg=1.0, iters=2000):
        self.lr = lr
        self.reg = reg
        self.iters = iters
        self.w = None
        self.b = None
        
    def cost(self, X, y):
        m = len(y)
        pred = X @ self.w + self.b
        err = pred - y
        base_cost = (1/(2*m)) * np.sum(err**2)
        penalty = (self.reg/(2*m)) * np.sum(self.w**2)
        return base_cost + penalty
    
    def fit(self, X, y):
        m, n = X.shape
        self.w = np.zeros(n)
        self.b = 0
        
        for i in range(self.iters):
            pred = X @ self.w + self.b
            err = pred - y
            
            dw = (1/m) * (X.T @ err) + (self.reg/m) * self.w
            db = (1/m) * np.sum(err)
            
            self.w -= self.lr * dw
            self.b -= self.lr * db
            
        return self
    
    def predict(self, X):
        return X @ self.w + self.b
    
    def score(self, X, y):
        pred = self.predict(X)
        ss_res = np.sum((y - pred)**2)
        ss_tot = np.sum((y - np.mean(y))**2)
        return 1 - (ss_res / ss_tot)

learn_rates = [0.0001, 0.001, 0.01, 0.1, 1, 10]
reg_vals = [1e-15, 1e-10, 1e-5, 1e-3, 0, 1, 10, 20]

res = []
for lr in learn_rates:
    for reg in reg_vals:
        mdl = MyRidge(lr=lr, reg=reg, iters=2000)
        mdl.fit(X_train, y_train)
        r2_train = mdl.score(X_train, y_train)
        r2_test = mdl.score(X_test, y_test)
        final_cost = mdl.cost(X_train, y_train)
        res.append({'lr': lr, 'reg': reg, 'r2_train': r2_train, 'r2_test': r2_test, 'cost': final_cost})

df_res = pd.DataFrame(res)
df_res = df_res.sort_values('r2_test', ascending=False)

print("Top 5 configs:")
print(df_res.head())
print("\nBest:")
best = df_res.iloc[0]
print(f"LR={best['lr']}, Reg={best['reg']}, R2={best['r2_test']:.4f}, Cost={best['cost']:.4f}")



       lr     alpha status  final_cost  mse_test   r2_test  iters
10  0.001   1.00000     ok    0.120167  0.219727  0.989340  20000
9   0.001   0.10000     ok    0.117177  0.219742  0.989339  20000
8   0.001   0.01000     ok    0.116878  0.219744  0.989339  20000
7   0.001   0.00100     ok    0.116848  0.219744  0.989339  20000
6   0.001   0.00001     ok    0.116844  0.219744  0.989339  20000
16  0.010   1.00000     ok    0.120081  0.219756  0.989338  20000
11  0.001  10.00000     ok    0.149976  0.219757  0.989338  20000
17  0.010  10.00000     ok    0.149970  0.219761  0.989338   8772
23  0.100  10.00000     ok    0.149970  0.219762  0.989338   1336
15  0.010   0.10000     ok    0.117040  0.219779  0.989337  20000


Q2: Hitters Dataset Regression

Load data from: https://gist.githubusercontent.com/keeganhines/59974f1ebef97bbaa44fb19143f90bad/raw/Hitters.csv

a) Clean the data - handle missing values and encode categorical variables
b) Split features and target, then scale
c) Train Linear, Ridge (alpha=0.5748), and Lasso (alpha=0.5748) models
d) Compare test performance and identify the best model

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_squared_error, r2_score

data_url = "https://gist.githubusercontent.com/keeganhines/59974f1ebef97bbaa44fb19143f90bad/raw/Hitters.csv"
hitters = pd.read_csv(data_url)

print(hitters.shape)
print(hitters.head())

hitters = hitters.dropna()
print(f"\nAfter dropping nulls: {hitters.shape}")

cats = hitters.select_dtypes(include='object').columns.tolist()
if 'Player' in hitters.columns:
    hitters = hitters.drop('Player', axis=1)
    if 'Player' in cats:
        cats.remove('Player')

hitters_encoded = pd.get_dummies(hitters, columns=cats, drop_first=True)

X = hitters_encoded.drop('Salary', axis=1)
y = hitters_encoded['Salary']

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=99)

scaler = StandardScaler()
X_tr_scaled = scaler.fit_transform(X_tr)
X_te_scaled = scaler.transform(X_te)

alpha_val = 0.5748

models = {
    'Linear': LinearRegression(),
    'Ridge': Ridge(alpha=alpha_val),
    'Lasso': Lasso(alpha=alpha_val, max_iter=5000)
}

for name, model in models.items():
    model.fit(X_tr_scaled, y_tr)
    preds = model.predict(X_te_scaled)
    mse = mean_squared_error(y_te, preds)
    r2 = r2_score(y_te, preds)
    print(f"\n{name}:")
    print(f"  MSE: {mse:.2f}")
    print(f"  RMSE: {np.sqrt(mse):.2f}")
    print(f"  R2: {r2:.4f}")


Shape: (322, 21)
          Unnamed: 0  AtBat  Hits  HmRun  Runs  RBI  Walks  Years  CAtBat  \
0     -Andy Allanson    293    66      1    30   29     14      1     293   
1        -Alan Ashby    315    81      7    24   38     39     14    3449   
2       -Alvin Davis    479   130     18    66   72     76      3    1624   
3      -Andre Dawson    496   141     20    65   78     37     11    5628   
4  -Andres Galarraga    321    87     10    39   42     30      2     396   

   CHits  ...  CRuns  CRBI  CWalks  League Division PutOuts  Assists  Errors  \
0     66  ...     30    29      14       A        E     446       33      20   
1    835  ...    321   414     375       N        W     632       43      10   
2    457  ...    224   266     263       A        W     880       82      14   
3   1575  ...    828   838     354       N        E     200       11       3   
4    101  ...     48    46      33       N        E     805       40       4   

   Salary  NewLeague  
0     NaN       

#### Cross Validation for Ridge and Lasso Regression

Explore Ridge Cross Validation (RidgeCV) and Lasso Cross Validation (LassoCV) function of Python. Implement both on Boston House Prediction Dataset (load_boston dataset from sklearn.datasets).

In [4]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import RidgeCV, LassoCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score

# --- Load dataset ---
try:
    from sklearn.datasets import load_boston
    boston = load_boston()
    X, y = boston.data, boston.target
except Exception:
    url = "http://lib.stat.cmu.edu/datasets/boston"
    raw = pd.read_csv(url, sep=r"\s+", header=None, skiprows=22)
    data = np.hstack([raw.values[::2, :], raw.values[1::2, :2]])
    X, y = data[:, :-1], data[:, -1]

# --- Scale and split ---
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

# --- Ridge Regression with CV ---
alpha_grid = np.logspace(-6, 6, 200)
ridge = RidgeCV(alphas=alpha_grid, store_cv_results=True)
ridge.fit(X_train, y_train)

ridge_preds = ridge.predict(X_test)
print(f"RidgeCV - best α = {ridge.alpha_:.6f}")
print(f"R² = {r2_score(y_test, ridge_preds):.4f} | MSE = {mean_squared_error(y_test, ridge_preds):.4f}")

# --- Lasso Regression with CV ---
lasso = LassoCV(cv=5, max_iter=10000)
lasso.fit(X_train, y_train)

lasso_preds = lasso.predict(X_test)
print(f"\nLassoCV - best α = {lasso.alpha_:.6f}")
print(f"R² = {r2_score(y_test, lasso_preds):.4f} | MSE = {mean_squared_error(y_test, lasso_preds):.4f}")

RidgeCV - best α = 26.126752
R² = 0.6905 | MSE = 16.0824

LassoCV - best α = 0.047088
R² = 0.6948 | MSE = 15.8604


#### Multiclass Logistic Regression:
Implement Multiclass Logistic Regression (step-by step) on Iris dataset using one vs. rest strategy?

In [5]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# --- Load data ---
iris = load_iris()
X, y = iris.data, iris.target

# --- Scale & split ---
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_tr, X_te, y_tr, y_te = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

# --- Sigmoid & training ---
def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

def fit_binary_logistic(X, y_bin, lr=0.1, steps=5000, tol=1e-6):
    m, n = X.shape
    Xb = np.c_[np.ones((m, 1)), X]
    w = np.zeros(n + 1)
    for _ in range(steps):
        preds = sigmoid(Xb @ w)
        grad = (1/m) * Xb.T.dot(preds - y_bin)
        w -= lr * grad
        if np.linalg.norm(grad) < tol:
            break
    return w

# --- Train one-vs-rest classifiers ---
K = len(np.unique(y_tr))
W = []
for k in range(K):
    y_k = (y_tr == k).astype(int)
    w_k = fit_binary_logistic(X_tr, y_k, lr=0.3, steps=10000)
    W.append(w_k)
W = np.vstack(W)

# --- Predict ---
def predict_multiclass(X, W):
    Xb = np.c_[np.ones((X.shape[0], 1)), X]
    probs = sigmoid(Xb @ W.T)
    return np.argmax(probs, axis=1)

y_pred = predict_multiclass(X_te, W)

# --- Evaluation ---
print(f"Accuracy: {accuracy_score(y_te, y_pred):.4f}\n")
print("Classification Report:")
print(classification_report(y_te, y_pred, target_names=iris.target_names))
print("Confusion Matrix:\n", confusion_matrix(y_te, y_pred))

Accuracy: 1.0000

Classification Report:
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        10
  versicolor       1.00      1.00      1.00         9
   virginica       1.00      1.00      1.00        11

    accuracy                           1.00        30
   macro avg       1.00      1.00      1.00        30
weighted avg       1.00      1.00      1.00        30

Confusion Matrix:
 [[10  0  0]
 [ 0  9  0]
 [ 0  0 11]]
